# Lab 3 — MLflow: Datasets and Evaluations

**Learning Goal**: Use MLflow to create a dataset, add entries to it, run model evaluations, and compare runs — so you can reuse these patterns in the next sessions.

---

## Setup

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

print('Imports OK')

In [ ]:
df = pd.read_csv(Path('../data/loan_applications.csv'))

le = LabelEncoder()
df['loan_purpose_enc'] = le.fit_transform(df['loan_purpose'])

FEATURES = [
    'credit_score', 'annual_income', 'loan_amount',
    'num_defaults', 'employment_years', 'age', 'loan_purpose_enc'
]
TARGET = 'approved'

idx_train, idx_test = train_test_split(
    df.index, test_size=0.20, random_state=42, stratify=df[TARGET]
)

X_train = df.loc[idx_train, FEATURES]
X_test  = df.loc[idx_test, FEATURES]
y_train = df.loc[idx_train, TARGET]
y_test  = df.loc[idx_test, TARGET]

# Evaluation data: features + target (required by mlflow.models.evaluate)
eval_data = X_test.copy()
eval_data[TARGET] = y_test.values

print(f'Train: {len(y_train)} | Test: {len(y_test)}')

In [ ]:
# Train a simple model (you'll log and evaluate it below)
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, verbosity=0)
model.fit(X_train, y_train)
print('Model trained')

---
## Section 1 — Creating a Dataset in MLflow

Create a dataset from the evaluation DataFrame using `mlflow.data.from_pandas()`. Use:
- `eval_data` as the DataFrame
- `source`: path to the CSV (e.g. `Path('../data/loan_applications.csv').resolve()` as string)
- `name`: e.g. `'loan_eval_test'`
- `targets`: the name of the target column (`TARGET`)

Then print the dataset's `name`, `digest`, and `schema`.

In [ ]:
data_path = Path('../data/loan_applications.csv').resolve()

# TODO: Create dataset with mlflow.data.from_pandas(eval_data, source=..., name=..., targets=...)
dataset = None  # replace with your call

# YOUR CODE HERE

# print(dataset.name, dataset.digest, dataset.schema)

---
## Section 2 — Logging a Run with Dataset and Evaluation

1. Set the experiment with `mlflow.set_experiment('lab03_mlflow_datasets_eval')`.
2. Start a run with `mlflow.start_run(run_name='xgb_baseline')`.
3. Log params: `model='XGBClassifier'`, `n_features=len(FEATURES)`.
4. Log the dataset: `mlflow.log_input(dataset, context='evaluation')`.
5. Log the model: use `infer_signature(X_test, model.predict(X_test))` and `mlflow.xgboost.log_model(model, artifact_path='model', signature=signature)`.
6. Run evaluation: `mlflow.models.evaluate(model_uri, eval_data, targets=TARGET, model_type='classifier', evaluators=['default'])` (use the `model_uri` from the log_model result).
7. Log the evaluation metrics: `mlflow.log_metrics(result.metrics)`.

In [ ]:
# TODO: Implement the run: set_experiment, start_run, log params, log_input(dataset), log_model, evaluate, log_metrics

# YOUR CODE HERE

# print('Run complete. Metrics:', result.metrics)

---
## Section 3 — Adding Entries to a Dataset

Datasets in MLflow are immutable. To "add entries":
1. Extend your DataFrame (e.g. concat `eval_data` with a slice of training data).
2. Create a new dataset with `from_pandas()` (e.g. name `'loan_eval_extended'`).
3. Start a new run, log this dataset with `log_input()`, log the same model, and run `evaluate()` on the extended DataFrame.

Example: add the first 200 training rows to `eval_data` with `pd.concat([eval_data, ...], ignore_index=True)`.

In [ ]:
# TODO: Build eval_data_extended = eval_data + first 200 rows of (X_train + y_train)
# Then create dataset_extended with from_pandas, and run a new MLflow run that logs it and evaluates the model on eval_data_extended

# YOUR CODE HERE

# print('Extended run complete. Metrics:', result.metrics)

---
## Section 4 — Comparing Runs Programmatically

Use `mlflow.search_runs()` to list all runs in the experiment and display a summary table of run names and metrics.

Hint: `client = mlflow.tracking.MlflowClient()`, `experiment = client.get_experiment_by_name('lab03_mlflow_datasets_eval')`, then `mlflow.search_runs(experiment_ids=[experiment.experiment_id])`.

In [ ]:
# TODO: Get experiment by name, search_runs, build a summary DataFrame with run names and metrics, print it

# YOUR CODE HERE

print('\n💡 Open the MLflow UI: run `mlflow ui` in your terminal, then visit http://localhost:5000')

---
## Key Takeaways

| Concept | What you learned |
|---|---|
| **Creating a dataset** | `mlflow.data.from_pandas(df, source=..., name=..., targets=...)` then `mlflow.log_input(dataset, context='evaluation')` |
| **Adding entries** | Extend the DataFrame (e.g. `pd.concat`), create a new dataset with `from_pandas`, log it in a new run |
| **Running evaluations** | `mlflow.models.evaluate(model_uri, eval_data, targets=..., model_type='classifier', evaluators=['default'])` and `mlflow.log_metrics(result.metrics)` |
| **Comparing runs** | `mlflow.search_runs(experiment_ids=[...])` to get a table of runs and metrics for use in later sessions |

**Next up**: Use these MLflow patterns (datasets + evaluations) in the next labs to compare models and track which data was used for each run.

---\n\n## Where MLflow stores data & using the UI\n\n**Storage (on disk, not in memory)**\n\n- **Tracking store**: By default MLflow writes run metadata (params, metrics, tags) to a folder `./mlruns` in the directory from which you run your code. A small SQLite DB and files live there, so runs persist after you close the notebook.\n- **Artifact store**: Logged models and other artifacts are stored under `mlruns/<experiment_id>/<run_id>/artifacts/`.\n- **Datasets**: When you call `log_input(dataset, context=...)`, MLflow stores only **dataset metadata** (name, digest, schema, profile), not the full DataFrame. The actual data stays in memory unless you also save it as an artifact (e.g. CSV) in the run.\n\n**Using the UI**\n\nFrom the directory that contains the `mlruns` folder (usually your project root), run in a terminal:\n\n```bash\nmlflow ui\n```\n\nThen open **http://localhost:5000** in your browser. You can browse experiments (e.g. `lab03_mlflow_datasets_eval`), compare runs, view metrics and params, see logged dataset inputs, and download or inspect artifacts (e.g. the logged model).